[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ThermoGraph/blob/main/examples/au_ni_miscibility.ipynb)

# Gold-Nickel Phase Diagram (Miscibility Gap)
This example demonstrates how to model a large solid miscibility gap using `zgraph`. We construct pure Gold and Nickel using `EinsteinNode`s, and model their highly repulsive solid solution by adding an intermediate interacting A-B microstate to the `FactorNode`.

In [ ]:
# Install ZGraph and ThermoGraph from PyPI if running in Google Colab
try:
    import zgraph as zg, thermograph as tg
except ImportError:
    !pip install -q zgraph thermograph
    import zgraph as zg, thermograph as tg

import jax
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

## 1. Physical Parameters
We define the properties of Au and Ni, including a strongly repulsive intermediate cluster state `Au-Ni`.

In [ ]:
T, mu_Au, mu_Ni = zg.SignalNodes(0, 1, 2)
RT = 8.314 * T

# 1. Pure Au FCC (Einstein)
au_e0_val = -10000.0
au_e0 = tg.GroundStateNode(au_e0_val)
au_osc = tg.EinsteinNode(165.0, T_index=0) # Au Debye temp
au_phase = au_e0 + 3.0 * au_osc

# 2. Pure Ni FCC (Einstein)
ni_e0_val = -15000.0
ni_e0 = tg.GroundStateNode(ni_e0_val)
ni_osc = tg.EinsteinNode(428.0, T_index=0) # Ni Debye temp
ni_phase = ni_e0 + 3.0 * ni_osc

# 3. Intermediate Au-Ni interacting pair (Einstein)
# We add a large positive mixing enthalpy (W) to simulate L0 > 0
W_mix = 12000.0  # Highly repulsive interaction
auni_e0 = tg.GroundStateNode(0.5 * au_e0_val + 0.5 * ni_e0_val + W_mix)
auni_osc = tg.EinsteinNode(0.5 * 165.0 + 0.5 * 428.0, T_index=0)
auni_phase = auni_e0 + 3.0 * auni_osc

## 2. Solid Solution with Miscibility Gap
We assemble the solid solution `FactorNode`. By adding the intermediate state with a multiplicity of 2 (representing A-B and B-A configurations), we create a Quasichemical-like solution.

In [ ]:
# The interaction state is a 50/50 mixture of Au and Ni atoms
w_au = mu_Au - au_phase
w_ni = mu_Ni - ni_phase
w_auni = zg.FactorNode([[0.5, 0.5, -1.0]], [mu_Au, mu_Ni, auni_phase])

# Solid FCC Solution with pair degeneracy g = [0, 0, ln(2)] for AB/BA configurations
M_solid = jnp.eye(3)
g_solid = jnp.array([0.0, 0.0, jnp.log(2.0)])
phase_FCC = zg.FactorNode(M_solid, [w_au, w_ni, w_auni], tau=RT, g=g_solid)

## 3. Extracting the Miscibility Gap
Because `PhaseBoundaryPredictor` finds *all* categorical crossovers, we can extract the FCC-FCC miscibility gap simply by treating the system as just the FCC phase! The predictor will find where the Au-rich state crosses the Ni-rich state.

In [ ]:
predictor_gap = tg.PhaseBoundaryPredictor(phase_FCC)
T_vals_gap = jnp.linspace(200, 1200, 100)
mu_diff_gap = jnp.linspace(-30000, 30000, 200)
T_g, mu_g = jnp.meshgrid(T_vals_gap, mu_diff_gap, indexing='ij')
inputs_gap = jnp.stack([T_g, mu_g/2, -mu_g/2], axis=-1)

# We extract the boundary (ranks 0 and 1 crossing indicates phase separation)
batched_x_gap = predictor_gap.predict_compositions(inputs_gap, mu_index=2)
x_left_gap = jnp.minimum(batched_x_gap[:, 0], batched_x_gap[:, 1])
x_right_gap = jnp.maximum(batched_x_gap[:, 0], batched_x_gap[:, 1])

fig_gap = go.Figure()
fig_gap.add_trace(go.Scatter(x=x_left_gap, y=T_vals_gap, mode='lines', name='Au-rich FCC', line=dict(color='goldenrod', width=3)))
fig_gap.add_trace(go.Scatter(x=x_right_gap, y=T_vals_gap, mode='lines', name='Ni-rich FCC', line=dict(color='silver', width=3)))

fig_gap.update_layout(title='Au-Ni Solid Miscibility Gap (Einstein Oscillators)', xaxis_title='Mole Fraction Ni', yaxis_title='Temperature (K)', width=800, height=600)
fig_gap.show()

## 4. Expanding to the Liquid Phase
We add dummy SGTE polynomials for the liquid phase to construct the full Au-Ni phase diagram.

In [ ]:
au_liq_node = tg.load_library('unary/GLIQAU')
ni_liq_node = tg.load_library('unary/GLIQNI')

# Ideal Liquid Solution
w_au_liq = mu_Au - au_liq_node
w_ni_liq = mu_Ni - ni_liq_node
phase_LIQ = zg.FactorNode(jnp.eye(2), [w_au_liq, w_ni_liq], tau=RT)

system = zg.FactorNode(jnp.eye(2), [phase_FCC, phase_LIQ], tau=0.0)

## 5. Full Phase Diagram
We extract the boundaries for the full system.

In [ ]:
predictor = tg.PhaseBoundaryPredictor(system)
T_vals_pd = jnp.linspace(800, 1800, 100)
T_g2, mu_g2 = jnp.meshgrid(T_vals_pd, mu_diff_gap, indexing='ij')
inputs_pd = jnp.stack([T_g2, mu_g2/2, -mu_g2/2], axis=-1)

batched_x = predictor.predict_compositions(inputs_pd, mu_index=2)
x_left = jnp.minimum(batched_x[:, 0], batched_x[:, 1])
x_right = jnp.maximum(batched_x[:, 0], batched_x[:, 1])

fig_pd = go.Figure()
fig_pd.add_trace(go.Scatter(x=x_left, y=T_vals_pd, mode='lines', name='Solidus', line=dict(color='red', width=3)))
fig_pd.add_trace(go.Scatter(x=x_right, y=T_vals_pd, mode='lines', name='Liquidus', line=dict(color='blue', width=3)))
fig_pd.update_layout(title='Full Au-Ni Phase Diagram', xaxis_title='Mole Fraction Ni', yaxis_title='Temperature (K)', width=800, height=600)
fig_pd.show()